# Coreference-Aware RAG Ingestion — LingMess Coref, Baseline vs Full Rewrite

**Question:** Does resolving coreference before chunking improve retrieval on pronoun-dependent queries?

**Chunking is held constant** across all variants (fixed word budget). The **only** thing that changes is the coreference strategy applied before indexing.

| Variant | Coref strategy | What gets embedded |
|---------|----------------|-------------------|
| **V0 – Baseline** | None | Original text |
| **V1 – Full rewrite** | LingMess resolves *all* mentions | Fully rewritten text |

Retrieval always **returns the original** chunk text; only the *embedded* text differs. Grading is **boundary-independent** — a hit means a returned chunk covers the gold *original-sentence index* — so both variants are directly comparable.

- **V0** establishes conventional-RAG retrieval performance (reference baseline).
- **V1** measures the benefit of automatic coreference resolution by enriching chunk semantics before indexing.

**Scope:** targeted stress test on **pronoun-dependent** questions — the case where coref matters most. Not a representative query mix.

**Coref:** `fastcoref` LingMessCoref (local, ~ms). **Q-gen:** `deepseek-ai/DeepSeek-V4-Flash` (DeepInfra). **Embeddings:** `Qwen/Qwen3-Embedding-8B` (DeepInfra).


## 1. Setup & Install

In [ ]:
!pip install -q openai datasets tqdm pandas numpy
# fastcoref's LingMessModel targets the transformers 4.x API. Kaggle ships transformers 5.x,
# whose new all_tied_weights_keys property breaks LingMess loading. Pin to 4.x.
!pip install -q "transformers<5" fastcoref
!python -m spacy download en_core_web_sm
# IMPORTANT: after this cell finishes, RESTART THE KERNEL once so the pinned transformers 4.x
# is the version actually imported, then run the cells below (skip re-running this one).
# NOTE: pip may print dependency-conflict WARNINGS about numba / cudf / cuml / dask-cuda
# (Kaggle's preinstalled RAPIDS GPU packages, unused here). Those are safe to ignore.


## 2. Config

Add a Kaggle Secret (Add-ons → Secrets), then attach it:
- `DEEPINFRA_TOKEN` — used for chat completions and embeddings.


In [ ]:
import os
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("coref_rag_benchmark")

try:
    from kaggle_secrets import UserSecretsClient
    DEEPINFRA_TOKEN = UserSecretsClient().get_secret("DEEPINFRA_TOKEN")
except Exception:
    DEEPINFRA_TOKEN = os.environ.get("DEEPINFRA_TOKEN", "")

assert DEEPINFRA_TOKEN, "Set DEEPINFRA_TOKEN as a Kaggle secret before running."

DEEPINFRA_BASE_URL = "https://api.deepinfra.com/v1/openai"
QGEN_MODEL = "deepseek-ai/DeepSeek-V4-Flash"
EMBED_MODEL = "Qwen/Qwen3-Embedding-8B"

# --- Document selection ---
DOC_MIN_WORDS = 3500
DOC_MAX_WORDS = 6000
MIN_PRONOUNS = 40

# --- Chunking: single fixed word budget (held constant across variants) ---
CHUNK_MAX_WORDS = 60         # pack whole sentences up to this many words

N_QUESTIONS = 20
TOP_K = 5

# Qwen3-Embedding uses an instruction prefix for QUERIES only.
QUERY_INSTRUCT = "Given a question, retrieve the passage that answers it"

log.info(
    "Config | doc %d-%d words | chunk<=%d words",
    DOC_MIN_WORDS, DOC_MAX_WORDS, CHUNK_MAX_WORDS,
)


In [ ]:
import time
import re
import json
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=DEEPINFRA_TOKEN, base_url=DEEPINFRA_BASE_URL)
log.info("DeepInfra client initialized | base_url=%s", DEEPINFRA_BASE_URL)

SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
PRONOUN_RE = re.compile(
    r"\b(he|she|him|her|his|hers|they|them|their|theirs|it|its)\b", re.IGNORECASE,
)

def split_sentences(text):
    return [s.strip() for s in SENT_SPLIT_RE.split(text.strip()) if s.strip()]

def chat(prompt, system=None, model=QGEN_MODEL, temperature=0.0, max_tokens=2000, task="chat"):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    t0 = time.time()
    resp = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens,
    )
    content = resp.choices[0].message.content
    log.info("[%s] %s | %.0fs", task, model, time.time() - t0)
    return content

def embed(texts, is_query=False, batch_size=32):
    """Return L2-normalized embeddings. Queries get the Qwen3 instruction prefix."""
    if is_query:
        texts = [f"Instruct: {QUERY_INSTRUCT}\nQuery: {t}" for t in texts]
    vecs = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(
            model=EMBED_MODEL, input=texts[i:i + batch_size], encoding_format="float",
        )
        vecs.extend(d.embedding for d in resp.data)
    arr = np.asarray(vecs, dtype="float32")
    arr /= (np.linalg.norm(arr, axis=1, keepdims=True) + 1e-9)
    return arr


## 2b. Coreference engine (LingMess, local)

We use `fastcoref`'s **LingMessCoref** model (higher accuracy than the default `FCoref`). It returns character-span clusters; we resolve a mention to the first **named** mention in its cluster (a span containing a proper noun), so pronouns become explicit entity names. Coref runs **locally in milliseconds** — no API latency.

`resolve_coref_text(text)` returns a rewritten string. The cell below runs a **manual sanity test** on hand-written pronoun-heavy snippets so you can eyeball quality before trusting it on real documents.


In [ ]:
# LingMess uses Longformer, whose sliding-window/global attention is incompatible with
# SDPA. transformers 5.x defaults to SDPA and, for Longformer, RAISES instead of falling
# back to eager (the requested impl == 'sdpa', so it re-raises). We must force eager.
#
# Passing attn_implementation as a kwarg to AutoModel.from_config is NOT reliable on 5.x:
# it does not always land on config._attn_implementation, which is what model init reads.
# So we set it directly on the config object (works on 4.x and 5.x) and keep the kwarg as
# a fallback. Guard flag lives on the AutoModel CLASS so re-running can never double-wrap.
from transformers.models.auto.modeling_auto import AutoModel

if not getattr(AutoModel, "_coref_eager_patched", False):
    _orig_from_config = AutoModel.from_config.__func__   # true original (we are not patched yet)

    @classmethod
    def _from_config_eager(cls, config, **kwargs):
        # Reliable across versions: force eager on the config the model init will read.
        try:
            config._attn_implementation = "eager"
        except Exception:
            pass
        kwargs.setdefault("attn_implementation", "eager")
        try:
            return _orig_from_config(cls, config, **kwargs)
        except (TypeError, ValueError):
            # Some versions reject passing config AND attn_implementation together;
            # config._attn_implementation set above is enough, so retry without the kwarg.
            kwargs.pop("attn_implementation", None)
            return _orig_from_config(cls, config, **kwargs)

    AutoModel.from_config = _from_config_eager
    AutoModel._coref_eager_patched = True

from fastcoref import LingMessCoref

coref_model = LingMessCoref(device="cuda" if os.environ.get("CUDA_VISIBLE_DEVICES") else "cpu")
log.info("LingMessCoref loaded")

PRONOUN_SET = {
    "he", "she", "him", "her", "his", "hers", "they", "them", "their", "theirs",
    "it", "its", "himself", "herself", "themselves", "itself",
}

def _is_pronoun_span(text):
    return text.strip().lower() in PRONOUN_SET

def _has_pronoun_token(text):
    return any(w.strip(".,;:'\"").lower() in PRONOUN_SET for w in text.split())

def _pick_representative(spans_text):
    """Choose the best 'name' for a cluster. Preference: spans with a proper-noun token and
    NO pronoun token; else any pronoun-free span; else None (so we never reinject a pronoun).
    Among candidates pick the shortest (avoids dragging in trailing clauses)."""
    proper = [s for s in spans_text
              if not _has_pronoun_token(s) and any(t[:1].isupper() for t in s.split())]
    clean = [s for s in spans_text if not _has_pronoun_token(s)]
    pool = proper or clean
    if not pool:
        return None
    return min(pool, key=len).strip()

def coref_edits(text):
    """Return a sorted list of (start, end, replacement) char-offset edits that replace
    pronoun mentions with their cluster's representative name."""
    if not text.strip():
        return []
    preds = coref_model.predict(texts=[text])
    if not preds:                      # LingMess returns [] if the text exceeds its length limit
        return []
    clusters = preds[0].get_clusters(as_strings=False)   # lists of (start, end) char offsets
    edits = []
    for cluster in clusters:
        spans_text = [text[s:e] for s, e in cluster]
        rep = _pick_representative(spans_text)
        if not rep:
            continue
        for (s, e), st in zip(cluster, spans_text):
            if _is_pronoun_span(st) and st.strip().lower() != rep.lower():
                repl = rep + "'s" if st.strip().lower() in {"his", "her", "its", "their", "hers", "theirs"} else rep
                edits.append((s, e, repl))
    return sorted(edits, key=lambda x: x[0])

def apply_edits(text, edits):
    """Apply (start, end, replacement) edits right-to-left so offsets stay valid."""
    out = text
    for s, e, repl in sorted(edits, key=lambda x: x[0], reverse=True):
        out = out[:s] + repl + out[e:]
    return out

def resolve_coref_text(text):
    """Replace pronoun mentions with their representative name."""
    return apply_edits(text, coref_edits(text))


### Manual sanity test — pronoun-heavy snippets

Eyeball these. Good coref should turn the **bold** pronouns into the right names. If these look wrong, do not trust LingMess on full documents.


In [ ]:
SANITY_TESTS = [
    # (snippet, what you'd hope to see)
    "Marie Curie won the Nobel Prize. She later won a second one, and her research changed physics.",
    "Alan Turing met John von Neumann in Princeton. He admired him, and they discussed computing.",
    "The company launched a product. It sold well, and its revenue doubled the next year.",
    "Ada Lovelace wrote notes on the engine. She saw its potential, and they became famous.",
    "When Napoleon reached the city, he ordered his troops to rest because they were exhausted.",
]

for snip in SANITY_TESTS:
    resolved = resolve_coref_text(snip)
    print("ORIG :", snip)
    print("COREF:", resolved)
    print("pronouns:", len(PRONOUN_RE.findall(snip)), "->", len(PRONOUN_RE.findall(resolved)))
    print("-" * 80)


## 3. Load one long continuous document

Stream Wikipedia until we find one biography-style article: **3500-6000 words** with **>=40 third-person pronouns** (dense coref chains).


In [ ]:
from datasets import load_dataset

log.info("Streaming Wikipedia for ONE long doc | %d-%d words | min_pronouns=%d",
         DOC_MIN_WORDS, DOC_MAX_WORDS, MIN_PRONOUNS)
raw = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

doc = None
scanned = 0
for row in raw:
    scanned += 1
    text = (row.get("text") or "").strip()
    if not text:
        continue
    n_words = len(text.split())
    if not (DOC_MIN_WORDS <= n_words <= DOC_MAX_WORDS):
        continue
    pronoun_count = len(PRONOUN_RE.findall(text))
    if pronoun_count < MIN_PRONOUNS:
        continue
    doc = {"doc_id": str(row["id"]), "title": row.get("title", ""),
           "text": text, "n_words": n_words, "pronoun_count": pronoun_count}
    log.info("Selected | title=%r | words=%d | pronouns=%d", doc["title"], n_words, pronoun_count)
    break

assert doc, f"No doc found after scanning {scanned}. Widen word range or lower MIN_PRONOUNS."

DOC_ID = doc["doc_id"]
ORIGINAL_TEXT = doc["text"]
ORIG_SENTS = split_sentences(ORIGINAL_TEXT)
print(f"Document: {doc['title']!r}")
print(f"Words: {doc['n_words']} | Pronouns: {doc['pronoun_count']} | Sentences: {len(ORIG_SENTS)} | Scanned: {scanned}")
print(ORIGINAL_TEXT[:600], "...")


## 4. Full-document coreference (LingMess, local) — produce V1 text

LingMess (a Longformer) has a max input length, so we resolve the document in **word-budgeted windows** of whole sentences (`COREF_WINDOW_WORDS`). Within each window we get char-offset edits and route each edit back to the sentence that contains it. This yields `V1_SENTS`, strictly 1:1 aligned with `ORIG_SENTS`, with pronoun mentions resolved. `V0` simply uses `ORIG_SENTS`.

Tradeoff: coref context is limited to within a window, so a very long-range antecedent may be missed. Raise `COREF_WINDOW_WORDS` for more context, but keep it under the model's length limit (empty predictions = too long).


In [ ]:
COREF_WINDOW_WORDS = 350   # keep each coref call well under LingMess's length limit

def resolve_sentences_aligned(orig_sents, window_words=COREF_WINDOW_WORDS):
    """Coref the doc in word-budgeted windows of whole sentences, route char-offset edits
    back to each sentence. Returns (coref_sents, n_edits), 1:1 aligned with orig_sents."""
    coref = list(orig_sents)
    total_edits = 0

    # Group sentence indices into windows under window_words.
    windows, cur, cur_w = [], [], 0
    for i, s in enumerate(orig_sents):
        w = len(s.split())
        if cur and cur_w + w > window_words:
            windows.append(cur)
            cur, cur_w = [], 0
        cur.append(i)
        cur_w += w
    if cur:
        windows.append(cur)

    for win in windows:
        # Build window text; record each sentence's (index, start, end) span within it.
        spans, pos, parts = [], 0, []
        for k, idx in enumerate(win):
            if k > 0:
                parts.append(" ")
                pos += 1
            start = pos
            parts.append(orig_sents[idx])
            pos += len(orig_sents[idx])
            spans.append((idx, start, pos))
        joined = "".join(parts)

        edits = coref_edits(joined)   # [] if window too long or no clusters
        total_edits += len(edits)

        by_sent = {}
        for (s, e, repl) in edits:
            for (idx, ss, se) in spans:
                if ss <= s and e <= se:
                    by_sent.setdefault(idx, []).append((s - ss, e - ss, repl))
                    break
        for idx, local_edits in by_sent.items():
            coref[idx] = apply_edits(orig_sents[idx], local_edits)
    return coref, total_edits

t0 = time.time()
V1_SENTS, N_EDITS = resolve_sentences_aligned(ORIG_SENTS)   # full rewrite
log.info("LingMess coref | %.1fs | %d edits", time.time() - t0, N_EDITS)
assert len(V1_SENTS) == len(ORIG_SENTS), "Alignment invariant broken"

pron_before = len(PRONOUN_RE.findall(" ".join(ORIG_SENTS)))
pron_after = len(PRONOUN_RE.findall(" ".join(V1_SENTS)))
changed = sum(1 for o, c in zip(ORIG_SENTS, V1_SENTS) if o != c)
print(f"Sentences: {len(ORIG_SENTS)} | changed by coref: {changed} | edits: {N_EDITS}")
print(f"Pronouns: {pron_before} -> {pron_after} ({100 * (1 - pron_after / max(pron_before, 1)):.0f}% reduction)")
for i, (o, c) in enumerate(zip(ORIG_SENTS, V1_SENTS)):
    if o != c:
        print(f"\nExample [{i}]\n ORIG: {o[:160]}\n V1:   {c[:160]}")
        break


## 5. Chunking (fixed word budget, held constant)

One chunking scheme for everyone: pack whole sentences until the next would exceed `CHUNK_MAX_WORDS`. Boundaries are computed **once** on original sentence indices and reused for both variants, so the only difference between V0/V1 is the *embedded* text (original vs full-coref). Retrieval always returns the **original** text.


In [ ]:
def word_budget_groups(sentences, max_words):
    """Pack whole sentences into groups up to max_words. Same rule for large and micro."""
    groups, cur, cur_w = [], [], 0
    for i, s in enumerate(sentences):
        w = len(s.split())
        if cur and cur_w + w > max_words:
            groups.append(cur)
            cur, cur_w = [], 0
        cur.append(i)
        cur_w += w
    if cur:
        groups.append(cur)
    return groups

def build_records(prefix, groups, return_sents, embed_sents):
    recs = []
    for i, g in enumerate(groups):
        recs.append({
            "chunk_id": f"{prefix}_{i}",
            "sent_indices": set(g),
            "return_text": " ".join(return_sents[j] for j in g),
            "embed_text": " ".join(embed_sents[j] for j in g),
        })
    return recs

chunk_groups = word_budget_groups(ORIG_SENTS, CHUNK_MAX_WORDS)

# Same boundaries for both variants; only embed_text differs.
v0_chunks = build_records("v0", chunk_groups, ORIG_SENTS, ORIG_SENTS)   # baseline (no coref)
v1_chunks = build_records("v1", chunk_groups, ORIG_SENTS, V1_SENTS)     # full coref rewrite

cw = [len(c["return_text"].split()) for c in v0_chunks]
print(f"Chunks: {len(v0_chunks)} | words avg={np.mean(cw):.1f} min={min(cw)} max={max(cw)}")
j = min(5, len(cw) - 1)
print("\nSample chunk — V0 (original, also returned to user):")
print(" ", v0_chunks[j]["embed_text"][:200])
print("Sample chunk — V1 (full coref, embedded):")
print(" ", v1_chunks[j]["embed_text"][:200])


## 6. Eval questions (pronoun-dependent ground truth)

Generate questions whose answer lives in a chunk that refers to the target entity by **pronoun only** (no name). These are exactly the queries coref is meant to help. The gold target is the set of **original sentence indices** covered by that chunk, so grading is independent of which variant we score.


In [ ]:
PROPER_NAME_RE = re.compile(r"\b([A-Z][a-z]+(?:\s+[A-Z][a-z'-]+)*)\b")


def entity_in_text(entity, text):
    if not entity or not text:
        return False
    return bool(re.search(rf"\b{re.escape(entity)}\b", text, re.I))


def infer_entity_from_rewrite(orig, v1, earlier_text):
    """Entity name LingMess inserted when rewriting pronouns in orig -> v1."""
    orig_names = set(PROPER_NAME_RE.findall(orig))
    for name in sorted(set(PROPER_NAME_RE.findall(v1)), key=len, reverse=True):
        if name in orig_names:
            continue
        if entity_in_text(name, earlier_text):
            return name
    for name in sorted(set(PROPER_NAME_RE.findall(v1)), key=len, reverse=True):
        if entity_in_text(name, orig):
            continue
        if entity_in_text(name, earlier_text):
            return name
    return None


def answer_snippet_for(chunk_text, entity):
    sents = [
        s for s in split_sentences(chunk_text)
        if PRONOUN_RE.search(s) and not entity_in_text(entity, s)
    ]
    return " ".join(sents)[:400] if sents else ""


def find_pronoun_targets(chunks, orig_sents, v1_sents, limit=N_QUESTIONS):
    """Pick (chunk, entity) pairs that satisfy the benchmark rules in code, not via LLM."""
    targets, seen = [], set()

    def add(idx, entity, source):
        ch = chunks[idx]
        ret = ch["return_text"]
        snippet = answer_snippet_for(ret, entity)
        if not snippet:
            return False
        key = (idx, entity.lower())
        if key in seen:
            return False
        seen.add(key)
        targets.append({
            "gold_chunk_index": idx,
            "entity_name": entity,
            "gold_indices": set(ch["sent_indices"]),
            "answer_snippet": snippet,
            "source": source,
        })
        return True

    # Primary: sentences where LingMess resolved a pronoun to a name from earlier context.
    for idx in range(1, len(chunks)):
        ch = chunks[idx]
        ret = ch["return_text"]
        earlier = " ".join(c["return_text"] for c in chunks[:idx])
        if not PRONOUN_RE.search(ret):
            continue
        for si in sorted(ch["sent_indices"]):
            orig, v1 = orig_sents[si], v1_sents[si]
            if not PRONOUN_RE.search(orig):
                continue
            entity = infer_entity_from_rewrite(orig, v1, earlier)
            if not entity or entity_in_text(entity, ret):
                continue
            if add(idx, entity, "coref"):
                if len(targets) >= limit:
                    return targets

    # Fallback: frequent proper names from earlier chunks, absent from the answer chunk.
    if len(targets) < limit:
        from collections import Counter
        for idx in range(1, len(chunks)):
            ch = chunks[idx]
            ret = ch["return_text"]
            earlier = " ".join(c["return_text"] for c in chunks[:idx])
            if not PRONOUN_RE.search(ret):
                continue
            freq = Counter(n for n in PROPER_NAME_RE.findall(earlier) if len(n) >= 5)
            for entity, _ in freq.most_common(30):
                if entity_in_text(entity, ret):
                    continue
                if add(idx, entity, "freq"):
                    if len(targets) >= limit:
                        return targets
    return targets


QGEN_SYSTEM_PROMPT = """You write retrieval-evaluation questions for a RAG benchmark.

For each target you receive an entity name and an answer passage (the passage uses pronouns for that entity).
Write one question per target that:
1. Names the entity explicitly (never by pronoun).
2. Is answerable from the passage once pronouns are resolved to that entity.

Return ONLY JSON array, same order as targets:
[{"question": "..."}, ...]"""

targets = find_pronoun_targets(v0_chunks, ORIG_SENTS, V1_SENTS, limit=N_QUESTIONS)
log.info("Pronoun targets found: %d (coref=%d freq=%d)",
         len(targets),
         sum(t["source"] == "coref" for t in targets),
         sum(t["source"] == "freq" for t in targets))
assert targets, "No pronoun-dependent targets in document — try another doc or lower MIN_PRONOUNS."

qgen_blocks = "\n\n".join(
    f"Target {i + 1}\nEntity: {t['entity_name']}\nAnswer passage:\n{t['answer_snippet']}"
    for i, t in enumerate(targets)
)
prompt = f"Document: {doc['title']}\n\n{qgen_blocks}"

raw_out = chat(prompt, system=QGEN_SYSTEM_PROMPT, model=QGEN_MODEL,
               temperature=0.3, max_tokens=2000, task="qgen")
cleaned = re.sub(r"^```(?:json)?|```$", "", raw_out.strip(), flags=re.MULTILINE).strip()
parsed = json.loads(cleaned)
if isinstance(parsed, dict):
    parsed = parsed.get("questions", [parsed])

eval_questions = []
for t, item in zip(targets, parsed):
    q = (item.get("question") or "").strip()
    if not q:
        log.warning("Skip q — empty question for chunk %d entity=%r", t["gold_chunk_index"], t["entity_name"])
        continue
    eval_questions.append({
        "question": q,
        "entity_name": t["entity_name"],
        "gold_indices": t["gold_indices"],
        "gold_chunk_index": t["gold_chunk_index"],
    })

log.info("Eval questions kept: %d / %d targets", len(eval_questions), len(targets))
assert eval_questions, "No eval questions after qgen — rerun qgen."
print(f"Using {len(eval_questions)} coref-dependent questions")
for q in eval_questions[:3]:
    print("-", q["question"][:100])


## 7. Evaluate the two variants (boundary-independent grading)

For each variant we embed its `embed_text`, embed the queries with the Qwen3 query instruction, take top-`K` by cosine, and score a **hit** if any returned chunk covers a gold original-sentence index. Grading uses sentence indices (not chunk text), so V0/V1 are directly comparable.


In [ ]:
def evaluate(chunks, questions, k=TOP_K, label="cond"):
    doc_mat = embed([c["embed_text"] for c in chunks])              # normalized
    q_mat = embed([q["question"] for q in questions], is_query=True)
    sims = q_mat @ doc_mat.T
    topk = np.argsort(-sims, axis=1)[:, :k]

    hits, rr, per_q = 0, [], []
    for qi, q in enumerate(questions):
        gold = q["gold_indices"]
        rank = None
        for pos, ci in enumerate(topk[qi]):
            if chunks[ci]["sent_indices"] & gold:
                rank = pos + 1
                break
        hit = rank is not None
        hits += int(hit)
        rr.append(1.0 / rank if hit else 0.0)
        per_q.append({
            "question": q["question"], "hit": hit, "rank": rank,
            "retrieved": [chunks[ci]["return_text"] for ci in topk[qi]],
        })
    n = len(questions)
    res = {"label": label, "recall_at_k": hits / n, "mrr": float(np.mean(rr)), "per_query": per_q}
    log.info("[%s] Recall@%d=%.3f MRR=%.3f (%d/%d)", label, k, res["recall_at_k"], res["mrr"], hits, n)
    return res

conditions = [
    ("V0_baseline", v0_chunks),
    ("V1_full_coref", v1_chunks),
]
all_results = {label: evaluate(chunks, eval_questions, label=label) for label, chunks in conditions}


## 8. Results


In [ ]:
import pandas as pd

v0_recall = all_results["V0_baseline"]["recall_at_k"]
summary = pd.DataFrame([
    {"variant": k, "n_chunks": len(v0_chunks),
     "recall_at_5": v["recall_at_k"], "mrr": v["mrr"],
     "delta_recall_vs_V0": v["recall_at_k"] - v0_recall}
    for k, v in all_results.items()
])
log.info("Results:\n%s", summary.to_string(index=False))
summary


## 9. Head-to-head: baseline (V0) vs full coref (V1)

In [ ]:
all_q = {q["question"] for q in eval_questions}
v0h = {r["question"] for r in all_results["V0_baseline"]["per_query"] if r["hit"]}
v1h = {r["question"] for r in all_results["V1_full_coref"]["per_query"] if r["hit"]}

recovered = v1h - v0h   # V0 miss -> V1 hit
hurt = v0h - v1h        # V0 hit -> V1 miss
both_fail = all_q - v0h - v1h

print(f"V0 baseline   Recall@5: {all_results['V0_baseline']['recall_at_k']:.3f}")
print(f"V1 full coref Recall@5: {all_results['V1_full_coref']['recall_at_k']:.3f}")
print(f"Recovered by coref (V0 miss -> V1 hit): {len(recovered)}")
print(f"Hurt by coref (V0 hit -> V1 miss):      {len(hurt)}")
print(f"Both fail:                              {len(both_fail)}")

if recovered:
    print("\n--- Recovered by coref ---")
    for qq in list(recovered)[:5]:
        print("  +", qq[:90])
if hurt:
    print("\n--- Hurt by coref ---")
    for qq in list(hurt)[:5]:
        print("  -", qq[:90])


## 10. Inspect a case: chunk text V0 vs V1


In [ ]:
q = eval_questions[0]
gi = q["gold_chunk_index"]
print("QUESTION:", q["question"])
print("\nReturned to user (original, both variants):")
print(" ", v0_chunks[gi]["return_text"])
print("\nEmbedded — V0 baseline:")
print(" ", v0_chunks[gi]["embed_text"])
print("\nEmbedded — V1 full coref:")
print(" ", v1_chunks[gi]["embed_text"])
print("\nPronouns  V0:", len(PRONOUN_RE.findall(v0_chunks[gi]["embed_text"])),
      "| V1:", len(PRONOUN_RE.findall(v1_chunks[gi]["embed_text"])))


## Notes — interpreting the two variants

**Setup:** chunking is fixed; only the coref strategy on the embedded text changes.

| Pattern | Meaning |
|---------|---------|
| V1 ≈ V0 | Coref doesn't move retrieval on these queries |
| V1 > V0 | Full coref rewrite helps (context enrichment wins) |
| V1 < V0 | Full coref hurts — wrong resolutions inject misleading names |

**Honest limitations:**
- One document, ~20 self-generated pronoun-dependent questions — anecdotal, high variance. For a real result, loop over many docs and run a paired significance test on per-query reciprocal ranks.
- Questions are generated from the chunks under test and are deliberately pronoun-only, which favors the coref variant. This measures coref's *best case*, not a representative query mix.
- Coref correctness is proxied by pronoun-count reduction; a wrong resolution injects a misleading name into the embedding, which this eval won't catch directly (but Section 10 lets you eyeball it).

**Cost:** LingMess runs locally in ~ms/doc — no API latency for coref. Only q-gen and embeddings hit the network.
